# Multi-Factor Strategy - Brazil

The study below applies factor investing strategies, seeking to create a resilient and consistent portfolio in adverse scenarios by combining factors. Factors were first developed by Eugene Fama and Kenneth French during the 1970s, when they refined the CAPM formula by adding the Size (SMB) and Value (HML) factors. Since then, several themes have been explored, leading to the creation of new factors; in this portfolio we use: Profitability, Value, Growth, Safety, Low Volatility and Momentum. 

In [ ]:
# Library imports

import pandas as pd
import numpy as np
import statsmodels.api as sm
import quantstats as qs
from scipy import stats
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

from datetime import date, timedelta
from  bizdays import Calendar, set_option

cal = Calendar.load('ANBIMA')
set_option('mode', 'pandas')

### Settings from earlier versions

These are files imported and processed from the Comdinheiro API that were worked on in earlier versions of this notebook. Dates are converted to the datetime.date format.

In [ ]:
# Imports data already collected to make the analysis easier

path = '.'
factors_folder = 'factors'
data_folder = 'data'

companies = pd.read_csv(path + "/" + data_folder + "/" + 'index_composition.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
prices_df = pd.read_csv(path + "/" + data_folder + "/" + 'prices.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
volume = pd.read_csv(path + "/" + data_folder + "/" + 'volume.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
benchmarks = pd.read_csv(path + "/" + data_folder + "/" + 'benchmarks.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
nefin = pd.read_csv(path + "/" + data_folder + "/" + 'nefin_factors.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')

# Market factors
market_value = pd.read_csv(path + "/" + data_folder + "/" + 'relative_market_value.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
momentum = pd.read_csv(path + "/" + factors_folder + "/" + 'momentum.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
reversal = pd.read_csv(path + "/" + factors_folder + "/" + 'reversal.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
volatility = pd.read_csv(path + "/" + factors_folder + "/" + 'volatility.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')

# Profitability factors
roe = pd.read_csv(path + "/" + factors_folder + "/" + 'roe.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
roa = pd.read_csv(path + "/" + factors_folder + "/" + 'roa.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
gpoa = pd.read_csv(path + "/" + factors_folder + "/" + 'gpoa.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
cfoa = pd.read_csv(path + "/" + factors_folder + "/" + 'cfoa.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
acc = pd.read_csv(path + "/" + factors_folder + "/" + 'acc.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')

# Growth factors
g_revenue = pd.read_csv(path + "/" + factors_folder + "/" + 'g_revenue.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
g_ebitda = pd.read_csv(path + "/" + factors_folder + "/" + 'g_ebitda.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
g_earnings = pd.read_csv(path + "/" + factors_folder + "/" + 'g_earnings.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')

# Leverage factors
debt_to_assets = pd.read_csv(path + "/" + factors_folder + "/" + 'debt_to_assets.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
debt_to_ebitda = pd.read_csv(path + "/" + factors_folder + "/" + 'debt_to_ebitda.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
liab_to_equity = pd.read_csv(path + "/" + factors_folder + "/" + 'liab_to_equity.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')

# Value factors
earnings_yield = pd.read_csv(path + "/" + factors_folder + "/" + 'earnings_yield.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
book_to_price = pd.read_csv(path + "/" + factors_folder + "/" + 'book_to_price.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
sales_to_price = pd.read_csv(path + "/" + factors_folder + "/" + 'sales_to_price.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
fcf_to_price = pd.read_csv(path + "/" + factors_folder + "/" + 'fcf_to_price.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
ebitda_ev = pd.read_csv(path + "/" + factors_folder + "/" + 'ebitda_ev.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')

# Magic Formula
roc_mf = pd.read_csv(path + "/" + factors_folder + "/" + 'roc_mf.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
earnings_yield_mf = pd.read_csv(path + "/" + factors_folder + "/" + 'earnings_yield_mf.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
debt_ebit_mf = pd.read_csv(path + "/" + factors_folder + "/" + 'debt_ebit_mf.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')

# Index rebalancing dates
rebalance_dates = sorted(companies.index.unique())

# Backtest dates
ini_date = cal.preceding(rebalance_dates[0] - timedelta(days = 1)).date()
end_date = date.today()

In [ ]:
# Data treatment to remove repeated tickers from the stock universe
# The main ticker is selected based on trading volume

ticker_codes = [tick[0:4] for tick in sorted(companies['Ticker'].unique())]
repeated_tickers = [tick for tick in ticker_codes if ticker_codes.count(tick) > 1]
repeated_tickers = list(dict.fromkeys(repeated_tickers))

vol_median = volume.ffill(limit=1).rolling(42).median()

for tick in repeated_tickers:
    rep = volume.columns[volume.columns.str.contains(tick)]
    ticker_filter = vol_median[rep].apply(lambda r: np.where(r == r.max(), r, np.nan), axis = 1)
    vol_median[rep] = pd.DataFrame(ticker_filter.tolist(), columns = rep, index = vol_median.index)

# Building the universe of available companies

The universe of investable companies was selected based on the criteria below:
- Being part of the IbrA (Brazil Broad Index) on the rebalancing date;
- Being active on the rebalancing date;
- Having liquidity above 1 million, calculated from the median traded volume over the last 42 days;
- Companies in the Financial and Utilities sectors are excluded (for the magic formula factors only).

In [ ]:
# Creates a matrix with the investable companies on each rebalancing date, showing the new index composition
investable = companies.loc[companies['Index'].isin(['SMLL', 'MLCX'])].pivot_table(index = 'Date', columns = 'Ticker', values = 'Weight')

# Multiplies by 0 and adds 1 to turn all values into 1, indicating that the company is investable
investable = investable.mul(0).add(1)

# Reindexes the table so that all backtest dates are present
investable = investable.reindex(market_value.index, method='ffill')
investable.loc[:date(2009,1,2)] = investable.loc[date(2009,1,5):date(2009,1,5)].values

# Creates a dataframe showing whether the stock had a price on the day, avoiding companies that left the market
has_price = prices_df.apply(lambda col: np.where((col.isna()) | (col <= 0), np.nan, 1)).loc[investable.index]

# Creates a dataframe to filter companies with low liquidity, avoiding companies whose prices are a problem to trade
volume_filter = vol_median.apply(lambda col: np.where(col > 1000, 1, np.nan))

In [ ]:
# Sector filter for the Magic Formula
# Excludes Financials and Utilities companies
mf_filter = companies.loc[~companies['Sector'].isin(['Financeiro', 'Utilidade Pública'])].pivot_table(index = 'Date', columns = 'Ticker', values = 'Weight')

# Multiplies by 0 and adds 1 to turn all values into 1, indicating that the company is investable
mf_filter = mf_filter.mul(0).add(1)

# Reindexes the table so that all backtest dates are present
mf_filter = mf_filter.reindex(market_value.index, method='ffill')
mf_filter.loc[:date(2009,1,2)] = mf_filter.loc[date(2009,1,5):date(2009,1,5)].values

In [ ]:
# Multiplies the indicators dataframe by investable and has_price to filter all dates at once
# Values multiplied by NaN become NaN, leaving only the values of companies that are investable on the date and have a price
roe = roe * investable * has_price * volume_filter
roa = roa * investable * has_price * volume_filter
gpoa = gpoa * investable * has_price * volume_filter
cfoa = cfoa * investable * has_price * volume_filter
acc = acc * investable * has_price * volume_filter

g_revenue = g_revenue * investable * has_price * volume_filter
g_ebitda = g_ebitda * investable * has_price * volume_filter
g_earnings = g_earnings * investable * has_price * volume_filter

debt_to_assets = debt_to_assets * investable * has_price * volume_filter
debt_to_ebitda = debt_to_ebitda * investable * has_price * volume_filter
liab_to_equity = liab_to_equity * investable * has_price * volume_filter

earnings_yield = earnings_yield * investable * has_price * volume_filter
book_to_price = book_to_price * investable * has_price * volume_filter
sales_to_price = sales_to_price * investable * has_price * volume_filter
fcf_to_price = fcf_to_price * investable * has_price * volume_filter
ebitda_ev = ebitda_ev * investable * has_price * volume_filter

momentum = momentum * investable * has_price * volume_filter
reversal = reversal * investable * has_price * volume_filter
volatility = volatility * investable * has_price * volume_filter
market_value = market_value * investable * has_price * volume_filter

roc_mf = roc_mf * investable * has_price * volume_filter * mf_filter
earnings_yield_mf = earnings_yield_mf * investable * has_price * volume_filter * mf_filter
debt_ebit_mf = debt_ebit_mf * investable * has_price * volume_filter * mf_filter

# Reindexes the dataframes so that all backtest dates are present
roe = roe.loc[volume.index, prices_df.columns]
roa = roa.loc[volume.index, prices_df.columns]
gpoa = gpoa.loc[volume.index, prices_df.columns]
cfoa = cfoa.loc[volume.index, prices_df.columns]
acc = acc.loc[volume.index, prices_df.columns]

g_revenue = g_revenue.loc[volume.index, prices_df.columns]
g_ebitda = g_ebitda.loc[volume.index, prices_df.columns]
g_earnings = g_earnings.loc[volume.index, prices_df.columns]

debt_to_assets = debt_to_assets.loc[volume.index, prices_df.columns]
debt_to_ebitda = debt_to_ebitda.loc[volume.index, prices_df.columns]
liab_to_equity = liab_to_equity.loc[volume.index, prices_df.columns]

earnings_yield = earnings_yield.loc[volume.index, prices_df.columns]
book_to_price = book_to_price.loc[volume.index, prices_df.columns]
sales_to_price = sales_to_price.loc[volume.index, prices_df.columns]
fcf_to_price = fcf_to_price.loc[volume.index, prices_df.columns]
ebitda_ev = ebitda_ev.loc[volume.index, prices_df.columns]

momentum = momentum.loc[volume.index, prices_df.columns]
reversal = reversal.loc[volume.index, prices_df.columns]
volatility = volatility.loc[volume.index, prices_df.columns]
market_value = market_value.loc[volume.index, prices_df.columns]

roc_mf = roc_mf.loc[volume.index, prices_df.columns]
earnings_yield_mf = earnings_yield_mf.loc[volume.index, prices_df.columns]
debt_ebit_mf = debt_ebit_mf.loc[volume.index, prices_df.columns]

## Z-Score and signal construction

After filtering which companies are available for investment, we apply a z-score to the indicators to build the factor signals. To remove outliers, we limit the z-score value to 2 standard deviations.

In [ ]:
# Calculates z-scores using numpy
z_roe = stats.zscore(roe.values, axis=1, nan_policy='omit')
z_roa = stats.zscore(roa.values, axis=1, nan_policy='omit')
z_gpoa = stats.zscore(gpoa.values, axis=1, nan_policy='omit')
z_cfoa = stats.zscore(cfoa.values, axis=1, nan_policy='omit')
z_acc = stats.zscore(acc.values, axis=1, nan_policy='omit')
z_g_revenue = stats.zscore(g_revenue.values, axis=1, nan_policy='omit')
z_g_ebitda = stats.zscore(g_ebitda.values, axis=1, nan_policy='omit')
z_g_earnings = stats.zscore(g_earnings.values, axis=1, nan_policy='omit')
z_debt_to_assets = stats.zscore(debt_to_assets.values, axis=1, nan_policy='omit')
z_debt_to_ebitda = stats.zscore(debt_to_ebitda.values, axis=1, nan_policy='omit')
z_liab_to_equity = stats.zscore(liab_to_equity.values, axis=1, nan_policy='omit')
z_ey = stats.zscore(earnings_yield.values, axis=1, nan_policy='omit')
z_bp = stats.zscore(book_to_price.values, axis=1, nan_policy='omit')
z_sales = stats.zscore(sales_to_price.values, axis=1, nan_policy='omit')
z_fcf = stats.zscore(fcf_to_price.values, axis=1, nan_policy='omit')
z_ev = stats.zscore(ebitda_ev.values, axis=1, nan_policy='omit')
z_mom = stats.zscore(momentum.values, axis=1, nan_policy='omit')
z_rev = stats.zscore(reversal.values, axis=1, nan_policy='omit')
z_vol = stats.zscore(volatility.values, axis=1, nan_policy='omit')
z_size = stats.zscore(market_value.values, axis=1, nan_policy='omit')
z_roc_mf = stats.zscore(roc_mf.values, axis=1, nan_policy='omit')
z_ey_mf = stats.zscore(earnings_yield_mf.values, axis=1, nan_policy='omit')
z_de_mf = stats.zscore(debt_ebit_mf.values, axis=1, nan_policy='omit')

# Replaces values outside the [-2, 2] range with NaN
z_roe = np.where((z_roe >= -2) & (z_roe <= 2), z_roe, np.nan)
z_roa = np.where((z_roa >= -2) & (z_roa <= 2), z_roa, np.nan)
z_gpoa = np.where((z_gpoa >= -2) & (z_gpoa <= 2), z_gpoa, np.nan)
z_cfoa = np.where((z_cfoa >= -2) & (z_cfoa <= 2), z_cfoa, np.nan)
z_acc = np.where((z_acc >= -2) & (z_acc <= 2), z_acc, np.nan)
z_g_revenue = np.where((z_g_revenue >= -2) & (z_g_revenue <= 2), z_g_revenue, np.nan)
z_g_ebitda = np.where((z_g_ebitda >= -2) & (z_g_ebitda <= 2), z_g_ebitda, np.nan)
z_g_earnings = np.where((z_g_earnings >= -2) & (z_g_earnings <= 2), z_g_earnings, np.nan)
z_debt_to_assets = np.where((z_debt_to_assets >= -2) & (z_debt_to_assets <= 2), z_debt_to_assets, np.nan)
z_debt_to_ebitda = np.where((z_debt_to_ebitda >= -2) & (z_debt_to_ebitda <= 2), z_debt_to_ebitda, np.nan)
z_liab_to_equity = np.where((z_liab_to_equity >= -2) & (z_liab_to_equity <= 2), z_liab_to_equity, np.nan)
z_ey = np.where((z_ey >= -2) & (z_ey <= 2), z_ey, np.nan)
z_bp = np.where((z_bp >= -2) & (z_bp <= 2), z_bp, np.nan)
z_sales = np.where((z_sales >= -2) & (z_sales <= 2), z_sales, np.nan)
z_fcf = np.where((z_fcf >= -2) & (z_fcf <= 2), z_fcf, np.nan)
z_ev = np.where((z_ev >= -2) & (z_ev <= 2), z_ev, np.nan)
z_mom = np.where((z_mom >= -2) & (z_mom <= 2), z_mom, np.nan)
z_rev = np.where((z_rev >= -2) & (z_rev <= 2), z_rev, np.nan)
z_vol = np.where((z_vol >= -2) & (z_vol <= 2), z_vol, np.nan)
z_size = np.where((z_size >= -2) & (z_size <= 2), z_size, np.nan)
z_roc_mf = np.where((z_roc_mf >= -2) & (z_roc_mf <= 2), z_roc_mf, np.nan)
z_ey_mf = np.where((z_ey_mf >= -2) & (z_ey_mf <= 2), z_ey_mf, np.nan)
z_de_mf = np.where((z_de_mf >= -2) & (z_de_mf <= 2), z_de_mf, np.nan)

# Converts the result back to a DataFrame, preserving index and columns
z_roe = pd.DataFrame(z_roe, columns=roe.columns, index=roe.index)
z_roa = pd.DataFrame(z_roa, columns=roa.columns, index=roa.index)
z_gpoa = pd.DataFrame(z_gpoa, columns=gpoa.columns, index=gpoa.index)
z_cfoa = pd.DataFrame(z_cfoa, columns=cfoa.columns, index=cfoa.index)
z_acc = pd.DataFrame(z_acc, columns=acc.columns, index=acc.index)
z_g_revenue = pd.DataFrame(z_g_revenue, columns=g_revenue.columns, index=g_revenue.index)
z_g_ebitda = pd.DataFrame(z_g_ebitda, columns=g_ebitda.columns, index=g_ebitda.index)
z_g_earnings = pd.DataFrame(z_g_earnings, columns=g_earnings.columns, index=g_earnings.index)
z_debt_to_assets = pd.DataFrame(z_debt_to_assets, columns=debt_to_assets.columns, index=debt_to_assets.index)
z_debt_to_ebitda = pd.DataFrame(z_debt_to_ebitda, columns=debt_to_ebitda.columns, index=debt_to_ebitda.index)
z_liab_to_equity = pd.DataFrame(z_liab_to_equity, columns=liab_to_equity.columns, index=liab_to_equity.index)
z_ey = pd.DataFrame(z_ey, columns=earnings_yield.columns, index=earnings_yield.index)
z_bp = pd.DataFrame(z_bp, columns=book_to_price.columns, index=book_to_price.index)
z_sales = pd.DataFrame(z_sales, columns=sales_to_price.columns, index=sales_to_price.index)
z_fcf = pd.DataFrame(z_fcf, columns=fcf_to_price.columns, index=fcf_to_price.index)
z_ev = pd.DataFrame(z_ev, columns=ebitda_ev.columns, index=ebitda_ev.index)
z_mom = pd.DataFrame(z_mom, columns=momentum.columns, index=momentum.index)
z_rev = pd.DataFrame(z_rev, columns=reversal.columns, index=reversal.index)
z_vol = pd.DataFrame(z_vol, columns=volatility.columns, index=volatility.index)
z_size = pd.DataFrame(z_size, columns=market_value.columns, index=market_value.index)
z_roc_mf = pd.DataFrame(z_roc_mf, columns=roc_mf.columns, index=roc_mf.index)
z_ey_mf = pd.DataFrame(z_ey_mf, columns=earnings_yield_mf.columns, index=earnings_yield_mf.index)
z_de_mf = pd.DataFrame(z_de_mf, columns=debt_ebit_mf.columns, index=debt_ebit_mf.index)

# Calculates the aggregate scores
z_profitability = z_roe + z_gpoa + z_acc + z_roa
z_growth = z_g_revenue + z_g_ebitda + z_g_earnings
z_safety = -z_debt_to_assets - z_debt_to_ebitda - z_liab_to_equity
z_value = z_ev + z_ey + z_bp + z_sales
z_mf = -stats.zscore(roc_mf.rank(axis=1, ascending=False) + 
                     earnings_yield_mf.rank(axis=1, ascending=False) + 
                     debt_ebit_mf.rank(axis=1, ascending=True) + 
                     volatility.rank(axis = 1, ascending = True) +
                     momentum.rank(axis=1, ascending=False), axis=1, nan_policy='omit')

# Converts the final result to a DataFrame
z_mf = pd.DataFrame(z_mf, columns=roc_mf.columns, index=roc_mf.index)

# Portfolio backtest

Backtest parameters:
- All stocks are rebalanced on the purchase dates;
- The signals used are those from the day before the rebalancing day;
- The backtest does not include brokerage, stock lending and slippage costs;
- Stocks are traded at the closing price of the rebalancing day;
- Stocks are allocated with equal weights;

In [ ]:
from brfactors.selection import get_quintiles, get_longshort_decile, get_longshort_amount
from brfactors.backtest import backtest_quintiles, backtest_longshort
from brfactors.dates import get_weekday_list

# Multi-Factor Strategy

The multi-factor strategy combines 3 different factors in order to build a portfolio that is resilient across several economic scenarios. The strategies are:
- Magic Formula: Created by Joel Greenblatt in 2006, it combines Return on Capital with Earnings Yield; the Net Debt/EBIT signal was added to the final ranking to control the exposure to highly leveraged companies;
- LowVol Momentum: Stocks with low volatility and a positive price trend;
- Quality: Combines indicators to select companies with high profitability, high growth, low leverage and low valuation.

<br>Backtest parameters:
- Backtest start: January 2012;
- Quarterly rebalancing (Jan, Apr, Jul, Oct);
- Each strategy has 10 stocks, the full portfolio has up to 30 stocks;
- Assets recommended by more than one strategy have their allocation increased.

In [ ]:
# Creates a list with all rebalancing dates
rebalance_dates = [date(year, month, 1) for year in range(2009, 2027) for month in range(1,13, 1)]

# Checks whether the date is a business day, if not, returns the next business day
rebalance_dates = [cal.following(ref_date) for ref_date in rebalance_dates if ref_date < date.today()]

In [ ]:
## Stock selection and backtest

# Stock selection
short_mf, long_mf = get_longshort_amount(20, z_mf) # Magic Formula
short_momvol, long_momvol = get_longshort_amount(10, -z_vol, use_filter = True, filter_signal = [z_mom], filter_percentile = [0.8]) # LowVol Momentum
short_quality, long_quality = get_longshort_amount(10, z_profitability + z_value, use_filter = True, filter_signal = [z_safety, z_growth], filter_percentile = [0.2, 0.2]) # Quality

# Backtest of the strategies
ret_mf = backtest_quintiles(prices_df, long_mf, rebalance_dates, market_value)
ret_momvol = backtest_quintiles(prices_df, long_momvol, rebalance_dates, market_value)
ret_quality = backtest_quintiles(prices_df, long_quality, rebalance_dates, market_value)
ret_comb = backtest_quintiles(prices_df, long_mf + long_momvol + long_quality, rebalance_dates, market_value)
# anti_ret_comb = backtest_quintiles(prices_df, short_mf + short_momvol + short_quality, rebalance_dates, market_value)
alpha_comb = (ret_comb - benchmarks['IBOV']).dropna()

In [ ]:
# Portfolio statistics

consolidated_returns = pd.concat([ret_mf, ret_momvol, ret_quality, ret_comb, alpha_comb, benchmarks['IBOV'], benchmarks['CDI']], axis=1).dropna(subset = 'Daily return')
consolidated_returns.columns = ['Magic Formula', 'LowVol Momentum', 'Quality', 'Combination', 'Combination-IBOV', 'IBOV', 'CDI']

total_return = consolidated_returns.add(1).prod().sub(1).map(lambda x: f'{x:.1%}')
annualized_return = consolidated_returns.add(1).prod().pow(252 / len(consolidated_returns)).sub(1).map(lambda x: f'{x:.1%}')
mean_return = consolidated_returns.mean().mul(252).map(lambda x: f'{x:.1%}')
annual_volatility = consolidated_returns.std().mul(np.sqrt(252)).map(lambda x: f'{x:.1%}')
sharpe_ratio = consolidated_returns.apply(lambda col: col - benchmarks['CDI'].loc[consolidated_returns.index]).mean().mul(252).div(consolidated_returns.std().mul(np.sqrt(252))).map(lambda x: f'{x:.2f}')
max_drawdown = consolidated_returns.apply(lambda col: qs.stats.max_drawdown(col)).map(lambda x: f'{x:.1%}')
ttest = pd.Series(stats.ttest_1samp(consolidated_returns.dropna(), 0)[0], index = consolidated_returns.columns).map(lambda x: f'{x:.2f}')

statistics = pd.DataFrame({
    'Total Return': total_return,
    'Annualized Return': annualized_return,
    'Mean Return': mean_return,
    'Volatility': annual_volatility,
    'Sharpe Ratio': sharpe_ratio,
    'Max Drawdown': max_drawdown,
    'T-Test': ttest
})

statistics.T

In [ ]:
# Historical performance

ret_mf.add(1).cumprod().plot(figsize = (20,8), title = 'Multi-Factor Strategy - Historical performance', label = 'Magic Formula')
# ret_momvol.add(1).cumprod().plot(figsize = (20,8), title = 'LowVol Momentum - Historical performance', label = 'Momentum + Low Vol')
# ret_quality.add(1).cumprod().plot(figsize = (20,8), title = 'Multi-Factor Strategy - Historical performance', label = 'Quality')
# ret_comb.add(1).cumprod().plot(figsize = (20,8), title = 'Multi-Factor Strategy - Historical performance', label = 'Combination')
# anti_ret_comb.add(1).cumprod().plot(figsize = (20,8), title = 'Multi-Factor Strategy - Historical performance', label = 'Inverse Combination')
# alpha_comb.add(1).cumprod().plot(figsize = (20,8), title = 'Multi-Factor Strategy - Historical performance', label = 'Alpha Combination')
benchmarks['IBOV'].loc[ret_comb.dropna().index].add(1).cumprod().plot(figsize = (20,8), title = 'Multi-Factor Strategy - Historical performance', label = 'IBOV')
# benchmarks['CDI'].loc[ret_comb.dropna().index].add(1).cumprod().plot(figsize = (20,8), title = 'Multi-Factor Strategy - Historical performance', label = 'CDI')

plt.legend()
plt.ylabel('Cumulative Return - NAV')
plt.xlabel('Date')
plt.yscale('log')

# Adjusts the number of ticks on the Y axis
plt.gca().yaxis.set_major_locator(ticker.LogLocator(base=10.0))  # Major ticks
plt.gca().yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs = 'auto'))  # Minor ticks

# Formats the Y axis values for better readability
plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter())
plt.gca().yaxis.set_minor_formatter(ticker.ScalarFormatter())

plt.gca().xaxis.set_major_locator(mdates.YearLocator(1))  # Major ticks every 1 year
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  # Formats the dates to show only the year

plt.grid(True, which='both')

In [ ]:
# Drawdowns

# qs.stats.to_drawdown_series(ret_mf.dropna()).plot(figsize = (20,8), title = 'Drawdown - Brazil Factors', label = 'Magic Formula')
qs.stats.to_drawdown_series(ret_momvol.dropna()).plot(figsize = (20,8), title = 'Drawdown - Brazil Factors', label = 'Momentum + Low Vol')
# qs.stats.to_drawdown_series(ret_quality.dropna()).plot(figsize = (20,8), title = 'Drawdown - Brazil Factors', label = 'Quality')
# qs.stats.to_drawdown_series(ret_comb.dropna()).plot(figsize = (20,8), title = 'Drawdown - Brazil Factors', label = 'Combination')
qs.stats.to_drawdown_series(benchmarks['IBOV'].loc[ret_comb.dropna().index]).plot(figsize = (20,8), title = 'Drawdown - LowVol Momentum', label = 'IBOV')
# qs.stats.to_drawdown_series(anti_ret_comb.dropna()).plot(figsize = (20,8), title = 'Drawdown - Brazil Factors', label = 'Inverse Combination')
# qs.stats.to_drawdown_series(alpha_comb.dropna()).plot(figsize = (20,8), title = 'Drawdown - Brazil Factors', label = 'Alpha Combination')

plt.legend()
plt.ylabel('Cumulative Return - NAV')
plt.xlabel('Date')

plt.gca().yaxis.set_major_formatter(ticker.PercentFormatter(1.0, decimals = 0))
plt.gca().xaxis.set_major_locator(mdates.YearLocator(1))  # Major ticks every 1 year

In [ ]:
# Monthly performance table of the strategy
qs.stats.monthly_returns(ret_momvol.dropna()).map('{:.2%}'.format)

In [ ]:
# Monthly performance table of the strategy
qs.stats.monthly_returns(benchmarks['IBOV'].loc[ret_comb.dropna().index]).map('{:.2%}'.format)

In [ ]:
# Factor exposure of the portfolio
returns_nefin = pd.merge(ret_momvol, nefin[['Rm_minus_Rf', 'Risk_Free', 'SMB', 'HML', 'WML']], how = 'left', left_index = True, right_index = True).dropna()
returns_nefin['Excess_Return'] = returns_nefin['Daily return'] - returns_nefin['Risk_Free']
monthly_returns = returns_nefin.resample('ME').apply(lambda x: (x + 1).prod() - 1)

Y = monthly_returns['Excess_Return']
X = monthly_returns[['Rm_minus_Rf', 'SMB', 'HML', 'WML']]
X = sm.add_constant(X)  # Adds a constant for the intercept
model = sm.OLS(Y, X).fit()
model.summary()

## Return per rebalancing

The analysis below seeks to understand whether allocating to the strategy was a good idea in the closed semester. Would it have been possible to make a better choice of parameters or stocks to improve the total return of the strategy? What influenced the performance of the negative windows? These and other questions can be studied through this breakdown.

In [ ]:
windows = pd.DataFrame({'start_date' : rebalance_dates, 'end_date' : rebalance_dates[1:] + [ret_comb.index[-1]]})
windows['Quarterly return'] = windows.apply(lambda row: ret_comb.loc[row['start_date'] : (row['end_date'] - pd.Timedelta(days=1))].add(1).prod() - 1, axis=1)
windows['IBRX Return'] = windows.apply(lambda row: benchmarks['IBRX'].loc[row['start_date'] : (row['end_date'] - pd.Timedelta(days=1))].add(1).prod() - 1, axis=1)
windows['MLCX Return'] = windows.apply(lambda row: benchmarks['MLCX'].loc[row['start_date'] : (row['end_date'] - pd.Timedelta(days=1))].add(1).prod() - 1, axis=1)
windows['SMLL Return'] = windows.apply(lambda row: benchmarks['SMLL'].loc[row['start_date'] : (row['end_date'] - pd.Timedelta(days=1))].add(1).prod() - 1, axis=1)

windows

## Strategy Trend Index

The Trend indicator shows how favorable the moment is for investing in the strategy; it is an aggregate of differences between moving averages, normalized by the volatility of each window.
The idea is to increase/reduce the exposure to the portfolio according to the indicator. Some variations of this strategy consist of:
- Reducing the gross of the Long/Short portfolio according to the signal.
- Switching between buying stocks or CDI in an 80/20 portfolio, ranging from 80 Stocks/20 CDI to 20 Stocks/80 CDI.

In [ ]:
# Cumulative return of the strategy
ret_strategy = ret_comb.add(1).cumprod()

# Exponential moving average (EWMA)
ewm_mean_5d = ret_strategy.ewm(span = 10).mean()
ewm_mean_21d = ret_strategy.ewm(span = 21).mean()
ewm_mean_63d = ret_strategy.ewm(span = 63).mean()
ewm_mean_126d = ret_strategy.ewm(span = 126).mean()
ewm_mean_252d = ret_strategy.ewm(span = 252).mean()

# Calculation of the spreads between the moving averages
spread_5d_21d = ewm_mean_5d - ewm_mean_21d
spread_21d_63d = ewm_mean_21d - ewm_mean_63d
spread_63d_126d = ewm_mean_63d - ewm_mean_126d
spread_126d_252d = ewm_mean_126d - ewm_mean_252d

# Normalization of the spreads by the standard deviation
norm_spread_5d_21d = spread_5d_21d / spread_5d_21d.rolling(252).std()
norm_spread_21d_63d = spread_21d_63d / spread_21d_63d.rolling(252).std()
norm_spread_63d_126d = spread_63d_126d / spread_63d_126d.rolling(252).std()
norm_spread_126d_252d = spread_126d_252d / spread_126d_252d.rolling(252).std()

# Trend signal calculation with the hyperbolic tangent
signal_5d_21d = np.tanh(norm_spread_5d_21d)
signal_21d_63d = np.tanh(norm_spread_21d_63d)
signal_63d_126d = np.tanh(norm_spread_63d_126d)
signal_126d_252d = np.tanh(norm_spread_126d_252d)

# Adjusted trend signal calculation: average of the signals
trend_signal = (signal_5d_21d + signal_21d_63d + signal_63d_126d + signal_126d_252d) / 4

trend_signal.plot(figsize = (20,10))

In [ ]:
trend_signal_adj = trend_signal.add(1).div(2)
trend_adj_ret = (ret_comb.sub(benchmarks['IBOV'].sub(benchmarks['CDI']).mul(trend_signal_adj.shift(1)))).dropna()
trend_adj_ret.add(1).cumprod().plot(figsize = (20,10), title = 'Historical return - Brazil Factors', label = 'Combination')

print(f'Total Return: {trend_adj_ret.add(1).prod() - 1:.2%}, CAGR: {trend_adj_ret.add(1).prod() ** (252/len(trend_adj_ret)) - 1:.2%}, Volatility: {trend_adj_ret.std() * np.sqrt(252):.2%}')

plt.legend()
plt.ylabel('Cumulative Return - NAV')
plt.xlabel('Date')
plt.yscale('log')

# Adjusts the number of ticks on the Y axis
plt.gca().yaxis.set_major_locator(ticker.LogLocator(base=10.0))  # Major ticks
plt.gca().yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs = 'auto'))  # Minor ticks

# Formats the Y axis values for better readability
plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter())
plt.gca().yaxis.set_minor_formatter(ticker.ScalarFormatter())

plt.gca().xaxis.set_major_locator(mdates.YearLocator(1))  # Major ticks every 1 year
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  # Formats the dates to show only the year

plt.grid(True, which='both')

# Further analyses

Complementary analyses that help to understand the behavior of the portfolio in different market scenarios and to improve the final performance.

## Portfolio return varying the number of stocks in the portfolio

The default number is 10 stocks per strategy; the test varies the number of stocks per strategy from 1 to 25 (the goal is to extrapolate the parameters to better understand the variations).

In [ ]:
returns_by_count = dict()

for n in tqdm(range(3,26)):
    # Stock selection
    short_mf, long_mf = get_longshort_amount(n, z_mf) # Magic Formula
    short_momvol, long_momvol = get_longshort_amount(n, -z_vol, use_filter = True, filter_signal = [z_mom], filter_percentile = [0.8]) # LowVol Momentum
    short_quality, long_quality = get_longshort_amount(n, z_profitability + z_value, use_filter = True, filter_signal = [z_safety, z_growth], filter_percentile = [0.2, 0.2]) # Quality

    # Backtest of the strategies
    ret_comb = backtest_quintiles(prices_df, long_mf + long_momvol + long_quality, rebalance_dates, market_value)

    returns_by_count[f'{int(n)} Stocks'] = ret_comb

returns_by_count = pd.DataFrame(returns_by_count).dropna()

In [ ]:
statistics = pd.DataFrame({'Annualized Return': returns_by_count.add(1).prod().pow(252 / len(returns_by_count)).sub(1),
                             'Mean Return': returns_by_count.mean().mul(252),
                             'Volatility': returns_by_count.std().mul(np.sqrt(252)),
                             'Sharpe Ratio': returns_by_count.apply(lambda col: col - benchmarks['CDI'].loc[returns_by_count.index]).mean().mul(252).div(returns_by_count.std().mul(np.sqrt(252))),
                             'Max Drawdown': returns_by_count.apply(lambda col: qs.stats.max_drawdown(col))})

statistics.sort_values(by = 'Sharpe Ratio', ascending = False).head()

In [ ]:
# Selects the 5 best numbers of stocks based on the Mean Return
top5 = statistics['Mean Return'].sort_values(ascending=False).head(5).index

fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# Left chart: Cumulative return of the 5 best strategies
(returns_by_count[top5].add(1).cumprod()).plot(ax=axes[0])
axes[0].set_title('Cumulative Return - Top 5 returns based on the number of stocks in the portfolio')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Cumulative Return')
axes[0].set_yscale('log')
axes[0].legend(title='Number of Stocks')

# Adjusts the number of ticks on the Y axis
axes[0].yaxis.set_major_locator(ticker.LogLocator(base=10.0))  # Major ticks
axes[0].yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs = 'auto'))  # Minor ticks
axes[0].yaxis.set_major_formatter(ticker.ScalarFormatter())
axes[0].yaxis.set_minor_formatter(ticker.ScalarFormatter())

axes[0].xaxis.set_major_locator(mdates.YearLocator(1))

axes[0].grid(True, which='both')

# Right chart: Mean return per stock
statistics['Mean Return'].plot(kind='bar', ax=axes[1])
axes[1].set_title('Annualized Mean Return by Number of Stocks')
axes[1].set_xlabel('Number of Stocks')
axes[1].set_ylabel('Annualized Mean Return')
axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(1.0, decimals = 0))


plt.tight_layout()
plt.show()

## Portfolio return varying the rebalancing period

The period used in the presentation was 3 months (locked to the first business day of January, April, July and October). The test varies how often the portfolio is rebalanced; the goal is to understand how the portfolio turnover frequency affects performance. This analysis is especially important when we add costs.
<p>The first test does not take reference dates into account, that is, it just counts N business days from one rebalancing to the next. Then the test is run using reference dates such as weekly (every Monday, for example), monthly, quarterly, semiannually and annually.

### Rebalancing without reference days (the portfolio is rebalanced every N business days)

In [ ]:
returns_by_interval = dict()

# Stock selection
short_mf, long_mf = get_longshort_amount(10, z_mf) # Magic Formula
short_momvol, long_momvol = get_longshort_amount(10, -z_vol, use_filter = True, filter_signal = [z_mom], filter_percentile = [0.8]) # LowVol Momentum
short_quality, long_quality = get_longshort_amount(10, z_profitability + z_value, use_filter = True, filter_signal = [z_safety, z_growth], filter_percentile = [0.2, 0.2]) # Quality 

for n in tqdm(range(1,253)):

    rebal_dates = prices_df.loc[date(2012,1,2):].index.tolist()[::n]
    # Backtest of the strategies
    ret_comb = backtest_quintiles(prices_df, long_mf + long_momvol + long_quality, rebal_dates, market_value)

    returns_by_interval[f'{n} Days'] = ret_comb

returns_by_interval = pd.DataFrame(returns_by_interval).dropna(how = 'all')

In [ ]:
statistics = pd.DataFrame({'Annualized Return': returns_by_interval.add(1).prod().pow(252 / len(returns_by_interval)).sub(1),
                             'Mean Return': returns_by_interval.mean().mul(252),
                             'Volatility': returns_by_interval.std().mul(np.sqrt(252)),
                             'Sharpe Ratio': returns_by_interval.apply(lambda col: col - benchmarks['CDI'].loc[returns_by_interval.index]).mean().mul(252).div(returns_by_interval.std().mul(np.sqrt(252))),
                             'Max Drawdown': returns_by_interval.apply(lambda col: qs.stats.max_drawdown(col))})

statistics.sort_values(by = 'Sharpe Ratio', ascending = False).head()

In [ ]:
# Selects the 5 best numbers of days based on the Mean Return
top5 = statistics['Mean Return'].sort_values(ascending=False).head(5).index

fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# Left chart: Cumulative return of the 5 best strategies
(returns_by_interval[top5].add(1).cumprod()).plot(ax=axes[0])
axes[0].set_title('Cumulative Return - Top 5 returns based on the number of stocks in the portfolio')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Cumulative Return')
axes[0].set_yscale('log')
axes[0].legend(title='Number of Days')

# Adjusts the number of ticks on the Y axis
axes[0].yaxis.set_major_locator(ticker.LogLocator(base=10.0))  # Major ticks
axes[0].yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs = 'auto'))  # Minor ticks
axes[0].yaxis.set_major_formatter(ticker.ScalarFormatter())
axes[0].yaxis.set_minor_formatter(ticker.ScalarFormatter())

axes[0].xaxis.set_major_locator(mdates.YearLocator(1))

axes[0].grid(True, which='both')

# Right chart: Mean return per stock
statistics['Mean Return'].plot(ax=axes[1])
axes[1].set_title('Annualized Mean Return by Number of Days')
axes[1].set_xlabel('Number of Days')
axes[1].set_ylabel('Annualized Mean Return')
axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(1.0, decimals = 0))


plt.tight_layout()
plt.show()

### Rebalancing with reference days
Rebalancing takes place on the next day if the reference date is a non-business day (weekends and holidays), returning to the reference day at the next rebalancing.

In [ ]:
# Last date with prices, the limit of all the lists below
last_date = prices_df.index[-1].date()

# List of dates by weekday for weekly rebalancing

monday = [cal.following(dt) for dt in get_weekday_list('first mon', 2012, 1, last_date)] # All Mondays that are business days from 2012 to the last date with prices
tuesday = [cal.following(dt) for dt in get_weekday_list('first tue', 2012, 1, last_date)]
wednesday = [cal.following(dt) for dt in get_weekday_list('first wed', 2012, 1, last_date)]
thursday = [cal.following(dt) for dt in get_weekday_list('first thu', 2012, 1, last_date)]
friday = [cal.following(dt) for dt in get_weekday_list('first fri', 2012, 1, last_date)]

# List of dates for monthly rebalancing
monthly = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in range(1,13) if cal.getdate('first bizday', year, month).date() < last_date]
mid_month_before = [cal.preceding(cal.getdate('14th day', year, month)) for year in range(2012, last_date.year + 1) for month in range(1,13) if cal.getdate('first bizday', year, month).date() < last_date]
mid_month_after = [cal.following(cal.getdate('16th day', year, month)) for year in range(2012, last_date.year + 1) for month in range(1,13) if cal.getdate('first bizday', year, month).date() < last_date]

# List of dates for bimonthly rebalancing
bimonthly_odd = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [1,3,5,7,9,11] if cal.getdate('first bizday', year, month).date() < last_date]
bimonthly_even = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [2,4,6,8,10,12] if cal.getdate('first bizday', year, month).date() < last_date]

# List of dates for quarterly rebalancing
quarterly_jan = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [1,4,7,10] if cal.getdate('first bizday', year, month).date() < last_date]
quarterly_feb = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [2,5,8,11] if cal.getdate('first bizday', year, month).date() < last_date]
quarterly_mar = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [3,6,9,12] if cal.getdate('first bizday', year, month).date() < last_date]

# List of dates for four-monthly rebalancing
fourmonthly_jan = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [1,5,9] if cal.getdate('first bizday', year, month).date() < last_date]
fourmonthly_feb = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [2,6,10] if cal.getdate('first bizday', year, month).date() < last_date]
fourmonthly_mar = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [3,7,11] if cal.getdate('first bizday', year, month).date() < last_date]
fourmonthly_apr = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [4,8,12] if cal.getdate('first bizday', year, month).date() < last_date]

# List of dates for semiannual rebalancing
semiannual_jan = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [1,7] if cal.getdate('first bizday', year, month).date() < last_date]
semiannual_feb = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [2,8] if cal.getdate('first bizday', year, month).date() < last_date]
semiannual_mar = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [3,9] if cal.getdate('first bizday', year, month).date() < last_date]
semiannual_apr = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [4,10] if cal.getdate('first bizday', year, month).date() < last_date]
semiannual_may = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [5,11] if cal.getdate('first bizday', year, month).date() < last_date]
semiannual_jun = [cal.getdate('first bizday', year, month) for year in range(2012, last_date.year + 1) for month in [6,12] if cal.getdate('first bizday', year, month).date() < last_date]

# List of dates for annual rebalancing
annual_jan = [cal.getdate('first bizday', year, 1) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 1).date() < last_date]
annual_feb = [cal.getdate('first bizday', year, 2) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 2).date() < last_date]
annual_mar = [cal.getdate('first bizday', year, 3) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 3).date() < last_date]
annual_apr = [cal.getdate('first bizday', year, 4) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 4).date() < last_date]
annual_may = [cal.getdate('first bizday', year, 5) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 5).date() < last_date]
annual_jun = [cal.getdate('first bizday', year, 6) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 6).date() < last_date]
annual_jul = [cal.getdate('first bizday', year, 7) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 7).date() < last_date]
annual_aug = [cal.getdate('first bizday', year, 8) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 8).date() < last_date]
annual_sep = [cal.getdate('first bizday', year, 9) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 9).date() < last_date]
annual_oct = [cal.getdate('first bizday', year, 10) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 10).date() < last_date]
annual_nov = [cal.getdate('first bizday', year, 11) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 11).date() < last_date]
annual_dec = [cal.getdate('first bizday', year, 12) for year in range(2012, last_date.year + 1) if cal.getdate('first bizday', year, 12).date() < last_date]

# Puts all lists together in a dictionary
rebalance_schedules = {
    'Weekly - Monday': monday,
    'Weekly - Tuesday': tuesday,
    'Weekly - Wednesday': wednesday,
    'Weekly - Thursday': thursday,
    'Weekly - Friday': friday,
    'Monthly': monthly,
    'Monthly - Before Mid-Month': mid_month_before,
    'Monthly - After Mid-Month': mid_month_after,
    'Bimonthly - Odd months': bimonthly_odd,
    'Bimonthly - Even months': bimonthly_even,
    'Quarterly - January': quarterly_jan,
    'Quarterly - February': quarterly_feb,
    'Quarterly - March': quarterly_mar,
    'Four-monthly - January': fourmonthly_jan,
    'Four-monthly - February': fourmonthly_feb,
    'Four-monthly - March': fourmonthly_mar,
    'Four-monthly - April': fourmonthly_apr,
    'Semiannual - January': semiannual_jan,
    'Semiannual - February': semiannual_feb,
    'Semiannual - March': semiannual_mar,
    'Semiannual - April': semiannual_apr,
    'Semiannual - May': semiannual_may,
    'Semiannual - June': semiannual_jun,
    'Annual - January': annual_jan,
    'Annual - February': annual_feb,
    'Annual - March': annual_mar,
    'Annual - April': annual_apr,
    'Annual - May': annual_may,
    'Annual - June': annual_jun,
    'Annual - July': annual_jul,
    'Annual - August': annual_aug,
    'Annual - September': annual_sep,
    'Annual - October': annual_oct,
    'Annual - November': annual_nov,
    'Annual - December': annual_dec
}

In [ ]:
returns_by_fixed_date = dict()

# Stock selection
short_mf, long_mf = get_longshort_amount(10, z_mf) # Magic Formula
short_momvol, long_momvol = get_longshort_amount(10, -z_vol, use_filter = True, filter_signal = [z_mom], filter_percentile = [0.8]) # LowVol Momentum
short_quality, long_quality = get_longshort_amount(10, z_profitability + z_value, use_filter = True, filter_signal = [z_safety, z_growth], filter_percentile = [0.2, 0.2]) # Quality

for n in tqdm(rebalance_schedules.keys()):
    # Selects the rebalancing dates
    rebal_dates = rebalance_schedules[n]
    # Backtest of the strategies
    ret_comb = backtest_quintiles(prices_df, long_mf + long_momvol + long_quality, rebal_dates, market_value)

    returns_by_fixed_date[n] = ret_comb

returns_by_fixed_date = pd.DataFrame(returns_by_fixed_date).dropna(how = 'all')

In [ ]:
statistics = pd.DataFrame({'Annualized Return': returns_by_fixed_date.add(1).prod().pow(252 / len(returns_by_fixed_date)).sub(1),
                             'Mean Return': returns_by_fixed_date.mean().mul(252),
                             'Volatility': returns_by_fixed_date.std().mul(np.sqrt(252)),
                             'Sharpe Ratio': returns_by_fixed_date.apply(lambda col: col - benchmarks['CDI'].loc[returns_by_fixed_date.index]).mean().mul(252).div(returns_by_fixed_date.std().mul(np.sqrt(252))),
                             'Max Drawdown': returns_by_fixed_date.apply(lambda col: qs.stats.max_drawdown(col))})

statistics.sort_values(by = 'Sharpe Ratio', ascending = False).head()

In [ ]:
# Selects the 5 best numbers of stocks based on the Mean Return
top5 = statistics['Mean Return'].sort_values(ascending=False).head(5).index

fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# Left chart: Cumulative return of the 5 best strategies
(returns_by_fixed_date[top5].add(1).cumprod()).plot(ax=axes[0])
axes[0].set_title('Cumulative Return - Top 5 returns based on the number of stocks in the portfolio')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Cumulative Return')
axes[0].set_yscale('log')
axes[0].legend(title='Number of Stocks')

# Adjusts the number of ticks on the Y axis
axes[0].yaxis.set_major_locator(ticker.LogLocator(base=10.0))  # Major ticks
axes[0].yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs = 'auto'))  # Minor ticks
axes[0].yaxis.set_major_formatter(ticker.ScalarFormatter())
axes[0].yaxis.set_minor_formatter(ticker.ScalarFormatter())

axes[0].xaxis.set_major_locator(mdates.YearLocator(1))

axes[0].grid(True, which='both')

# Right chart: Mean return per stock
statistics['Mean Return'].plot(ax=axes[1], kind = 'bar')
axes[1].set_title('Annualized Mean Return by Number of Stocks')
axes[1].set_xlabel('Number of Stocks')
axes[1].set_ylabel('Annualized Mean Return')
axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(1.0, decimals = 0))


plt.tight_layout()
plt.show()